# VRA/ANAC — ETL de 2020
## Etapas 1 e 2: deduplicação exata e separação de elegíveis e rejeitados

**Entrada:** os 12 CSVs de `dados/brutos/2020/`. **Referência de diagnóstico:** `notebooks/exploracao/exploracao_2020.ipynb`.

Mantemos `df_bruta` integral, deduplicamos exclusivamente pelas 20 colunas originais e registramos os descartes em `df_duplicatas`. Em seguida, separamos `df_sem_duplicatas` em `df_elegiveis` e `df_rejeitados`, com todos os motivos em `df_motivos_rejeicao`. Os valores originais e a rastreabilidade são preservados em todas as saídas.

**Modelagem consultada:** *Documento de Modelagem — Tabela Fato_voo*, versão 1.0, VRA/ANAC 2019–2021, arquivo `Documento_Tabela_Fato_VRA_ANAC_2019_2021.pdf` fornecido pelo integrante. A seção 4, página 2, estabelece os cinco campos obrigatórios e a auditoria de rejeições; a seção 6, página 3, exige medir cada ano e evitar sobreposição com duplicatas. O PDF se apresenta como especificação para aprovação do grupo; não presumimos que as pendências da seção 7 já foram resolvidas. O contrato Markdown e o esquema Draw.io citados no README ainda não estão no checkout.

**Escopo desta implementação:** somente deduplicação e separação por ausência/invalidade dos cinco campos obrigatórios. Não padronizamos os valores de saída, não transformamos datas na base, não calculamos métricas e não carregamos o PostgreSQL. As validações de datas usam apenas objetos temporários, sem converter as colunas originais.

### 1. Preparar o ambiente e localizar a entrada
Execute as células em ordem a partir da raiz do projeto ou de `notebooks/etl/`. São necessários Pandas e IPython, disponíveis no ambiente dos notebooks. Não há instalações, acesso à rede ou banco.

A raiz é localizada nos diretórios ancestrais. Exigimos os 12 nomes mensais exatos. Os hashes SHA-256 permitem conferir ao final que os arquivos continuam com os mesmos bytes.

In [1]:
from pathlib import Path
from collections import Counter
import csv
import hashlib
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 25)

raiz = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dados/brutos/2020").is_dir()),
    None,
)
if raiz is None:
    raise FileNotFoundError("Execute dentro do repositório que contém dados/brutos/2020/.")

pasta_bruta = raiz / "dados/brutos/2020"
arquivos = sorted(p for p in pasta_bruta.iterdir() if p.suffix.lower() == ".csv")
nomes_esperados = {f"VRA_2020_{mes:02d}.csv" for mes in range(1, 13)}
nomes_encontrados = {p.name for p in arquivos}
assert len(arquivos) == 12 and nomes_encontrados == nomes_esperados, (
    f"Arquivos ausentes: {sorted(nomes_esperados - nomes_encontrados)}; "
    f"inesperados: {sorted(nomes_encontrados - nomes_esperados)}"
)

def sha256_arquivo(caminho):
    resumo = hashlib.sha256()
    with caminho.open("rb") as fonte:
        for bloco in iter(lambda: fonte.read(1024 * 1024), b""):
            resumo.update(bloco)
    return resumo.hexdigest()

hashes_iniciais = {p.name: sha256_arquivo(p) for p in arquivos}
print("Pandas:", pd.__version__)
print("Arquivos encontrados:", len(arquivos))

Pandas: 3.1.1
Arquivos encontrados: 12


### 2. Ler as 20 colunas originais e adicionar rastreabilidade
Usamos `sep=";"`, UTF-8 com BOM opcional e leitura estrita, sem tentar outros encodings silenciosamente. `dtype=str`, `keep_default_na=False` e `na_filter=False` preservam números de voo com zeros à esquerda, vazios e marcadores textuais. Não aplicamos `strip()`, mudança de caixa ou conversão de datas.

Antes da leitura com Pandas, conferimos o cabeçalho literal e a largura de todos os registros com `csv.reader`. Isso impede que um cabeçalho divergente ou registro curto seja aceito silenciosamente. `skip_blank_lines=False` impede o descarte implícito de registros vazios.

`arquivo_origem` é o nome do CSV. `linha_origem` é a posição do registro lógico, começando em 2 após o cabeçalho; com quebras de linha dentro de campos citados, pode diferir da linha física do arquivo. A ordem é determinística: nome do arquivo crescente e posição original do registro.

In [2]:
colunas_originais = [
    "Sigla ICAO Empresa Aérea", "Empresa Aérea", "Número Voo", "Código DI",
    "Código Tipo Linha", "Modelo Equipamento", "Número de Assentos",
    "Sigla ICAO Aeroporto Origem", "Descrição Aeroporto Origem", "Partida Prevista",
    "Partida Real", "Sigla ICAO Aeroporto Destino", "Descrição Aeroporto Destino",
    "Chegada Prevista", "Chegada Real", "Situação Voo", "Justificativa",
    "Referência", "Situação Partida", "Situação Chegada",
]
colunas_origem = ["arquivo_origem", "linha_origem"]
partes = []
inventario = []

for arquivo in arquivos:
    with arquivo.open(encoding="utf-8-sig", newline="") as fonte:
        leitor = csv.reader(fonte, delimiter=";", strict=True)
        cabecalho = next(leitor, None)
        assert cabecalho == colunas_originais, f"Cabeçalho divergente: {arquivo.name}"
        larguras = Counter(len(registro) for registro in leitor)
        assert all(largura == 20 for largura in larguras), (
            f"Registro com largura divergente em {arquivo.name}: {dict(larguras)}"
        )
        registros_csv = sum(larguras.values())

    mensal = pd.read_csv(
        arquivo, sep=";", encoding="utf-8-sig", dtype=str,
        keep_default_na=False, na_filter=False, skip_blank_lines=False,
    )
    assert mensal.columns.tolist() == colunas_originais
    assert len(mensal) == registros_csv, f"Leitura incompleta: {arquivo.name}"
    mensal["arquivo_origem"] = arquivo.name
    mensal["linha_origem"] = range(2, len(mensal) + 2)
    partes.append(mensal)
    inventario.append({
        "arquivo": arquivo.name, "registros_brutos": len(mensal),
        "colunas_originais": 20, "sha256": hashes_iniciais[arquivo.name],
    })

df_bruta = pd.concat(partes, ignore_index=True)
resumo_arquivos = pd.DataFrame(inventario)
del partes, mensal
assert df_bruta.columns.tolist() == colunas_originais + colunas_origem
assert not df_bruta[colunas_originais].isna().any().any()
assert not df_bruta.duplicated(subset=colunas_origem).any()

def assinatura_dataframe(df):
    valores_hash = pd.util.hash_pandas_object(df, index=True).values
    return hashlib.sha256(valores_hash.tobytes()).hexdigest()

assinatura_bruta = assinatura_dataframe(df_bruta)
print("Registros brutos:", len(df_bruta))
print("Colunas originais:", len(colunas_originais), "| Com rastreabilidade:", len(df_bruta.columns))
display(resumo_arquivos)
display(df_bruta.head(3))

Registros brutos: 483545
Colunas originais: 20 | Com rastreabilidade: 22


,arquivo,registros_brutos,colunas_originais,sha256
0,VRA_2020_01.csv,92390,20,c64acd1a13397c83861acf5693bf60aa5550acf1fbad08...
1,VRA_2020_02.csv,80919,20,ba6a91885d1f1ded54cb46bb714d5a91b26294088a4f82...
2,VRA_2020_03.csv,64523,20,e6fe0059e5d784bf7e551bf14aa3e551771c4450dd047c...
3,VRA_2020_04.csv,7617,20,c76f5ea70b68896f06335eb54903e2b75cc57ee9b41b6e...
4,VRA_2020_05.csv,9514,20,20dfa011cb3423f92674baa629429dfb88b9950fa0a7c4...
5,VRA_2020_06.csv,12800,20,d01012074bcb98932f85e0c8ef76e02a2b1c0fc4a52200...
6,VRA_2020_07.csv,19850,20,883b2fc9da868733968ae13c03eb6da42834797b8acbfa...
7,VRA_2020_08.csv,24806,20,2c187990075bc37701abc3e1a0c4c0f6951cd928b1d34d...
8,VRA_2020_09.csv,30660,20,f23a155467ca9424ce6f74cf0acf4182650293aff0fabd...
9,VRA_2020_10.csv,40590,20,d12483fe7b950ace85c012c68ded986a9b8281018eaea2...


,Sigla ICAO Empresa Aérea,Empresa Aérea,Número Voo,Código DI,Código Tipo Linha,Modelo Equipamento,Número de Assentos,Sigla ICAO Aeroporto Origem,Descrição Aeroporto Origem,Partida Prevista,Partida Real,Sigla ICAO Aeroporto Destino,Descrição Aeroporto Destino,Chegada Prevista,Chegada Real,Situação Voo,Justificativa,Referência,Situação Partida,Situação Chegada,arquivo_origem,linha_origem
0,AAL,"AMERICAN AIRLINES, INC.",0213,1,I,B752,173,KMIA,"MIAMI INTERNATIONAL AIRPORT - MIAMI, FLORIDA -...",01/01/2020 18:47,01/01/2020 18:47,SBBR,PRESIDENTE JUSCELINO KUBITSCHEK - BRASÍLIA - D...,02/01/2020 02:27,02/01/2020 02:27,REALIZADO,,2020-01-01,Pontual,Pontual,VRA_2020_01.csv,2
1,AAL,"AMERICAN AIRLINES, INC.",0216,0,I,B789,285,SBGR,GUARULHOS - GOVERNADOR ANDRÉ FRANCO MONTORO - ...,01/01/2020 22:30,01/01/2020 22:30,KLAX,LOS ANGELES INTERNATIONAL AIRPORT - LOS ANGELE...,02/01/2020 11:00,02/01/2020 11:00,REALIZADO,,2020-01-01,Pontual,Pontual,VRA_2020_01.csv,3
2,AAL,"AMERICAN AIRLINES, INC.",0904,0,I,B772,273,SBGL,AEROPORTO INTERNACIONAL DO RIO DE JANEIRO (GAL...,01/01/2020 22:25,01/01/2020 22:25,KMIA,"MIAMI INTERNATIONAL AIRPORT - MIAMI, FLORIDA -...",02/01/2020 07:00,02/01/2020 07:00,REALIZADO,,2020-01-01,Pontual,Pontual,VRA_2020_01.csv,4


### 3. Deduplicar e registrar cada descarte
A comparação usa exclusivamente `colunas_originais`. As duas colunas de rastreabilidade não fazem parte da chave de igualdade. Usamos `keep="first"`: preserva a primeira ocorrência na ordem de leitura, sem presumir que seja a mais atual ou correta. Como os 20 valores são iguais, a escolha apenas fixa uma origem reproduzível para auditoria.

`df_bruta` permanece intacto. `df_sem_duplicatas` é uma cópia com as mesmas 22 colunas. `df_duplicatas` contém os 20 valores descartados, sua origem, `arquivo_origem_mantido`, `linha_origem_mantida` e o motivo. O vínculo é feito pela igualdade completa dos valores, com validação muitos-para-um, sem usar hashes como chave de deduplicação.

O registro de auditoria é um DataFrame reproduzível; a relação completa de origens descartadas e mantidas ficará nas saídas salvas do notebook. Esta etapa não exporta CSVs nem grava tabelas no banco.

In [3]:
def deduplicar_com_auditoria(base, colunas_dados):
    """Retorna base deduplicada e descartes ligados à primeira ocorrência."""
    excedentes = base.duplicated(subset=colunas_dados, keep="first")
    envolvidos = base.duplicated(subset=colunas_dados, keep=False)
    sem_duplicatas = base.loc[~excedentes].copy()

    # Apenas representantes dos grupos repetidos precisam participar da junção.
    representantes = base.loc[
        envolvidos & ~excedentes, colunas_dados + colunas_origem
    ].rename(columns={
        "arquivo_origem": "arquivo_origem_mantido",
        "linha_origem": "linha_origem_mantida",
    })
    descartadas = base.loc[excedentes].copy()
    auditoria = descartadas.merge(
        representantes, on=colunas_dados, how="left", sort=False, validate="many_to_one",
    )
    auditoria["motivo_descarte"] = "DUPLICATA_EXATA_20_COLUNAS"
    return sem_duplicatas, auditoria


df_sem_duplicatas, df_duplicatas = deduplicar_com_auditoria(df_bruta, colunas_originais)
mascara_excedentes = df_bruta.duplicated(subset=colunas_originais, keep="first")
mascara_envolvidos = df_bruta.duplicated(subset=colunas_originais, keep=False)
print("Registros envolvidos em grupos duplicados:", int(mascara_envolvidos.sum()))
print("Duplicatas excedentes descartadas:", len(df_duplicatas))
print("Registros após deduplicação:", len(df_sem_duplicatas))

colunas_vinculo = colunas_origem + [
    "arquivo_origem_mantido", "linha_origem_mantida", "motivo_descarte",
]
# Exibir todas as origens torna os 124 descartes consultáveis sem reexecutar.
with pd.option_context("display.max_rows", None):
    display(df_duplicatas[colunas_vinculo])
print("Amostra dos valores originais descartados e sua ocorrência mantida:")
display(df_duplicatas.head(3))

Registros envolvidos em grupos duplicados: 248
Duplicatas excedentes descartadas: 124
Registros após deduplicação: 483421
Amostra dos valores originais descartados e sua ocorrência mantida:


,arquivo_origem,linha_origem,arquivo_origem_mantido,linha_origem_mantida,motivo_descarte
0,VRA_2020_01.csv,11522,VRA_2020_01.csv,11521,DUPLICATA_EXATA_20_COLUNAS
1,VRA_2020_01.csv,90942,VRA_2020_01.csv,90941,DUPLICATA_EXATA_20_COLUNAS
2,VRA_2020_01.csv,90944,VRA_2020_01.csv,90943,DUPLICATA_EXATA_20_COLUNAS
3,VRA_2020_01.csv,92358,VRA_2020_01.csv,92357,DUPLICATA_EXATA_20_COLUNAS
4,VRA_2020_02.csv,561,VRA_2020_02.csv,560,DUPLICATA_EXATA_20_COLUNAS
5,VRA_2020_02.csv,563,VRA_2020_02.csv,562,DUPLICATA_EXATA_20_COLUNAS
6,VRA_2020_02.csv,565,VRA_2020_02.csv,564,DUPLICATA_EXATA_20_COLUNAS
7,VRA_2020_02.csv,567,VRA_2020_02.csv,566,DUPLICATA_EXATA_20_COLUNAS
8,VRA_2020_02.csv,569,VRA_2020_02.csv,568,DUPLICATA_EXATA_20_COLUNAS
9,VRA_2020_02.csv,571,VRA_2020_02.csv,570,DUPLICATA_EXATA_20_COLUNAS


,Sigla ICAO Empresa Aérea,Empresa Aérea,Número Voo,Código DI,Código Tipo Linha,Modelo Equipamento,Número de Assentos,Sigla ICAO Aeroporto Origem,Descrição Aeroporto Origem,Partida Prevista,Partida Real,Sigla ICAO Aeroporto Destino,Descrição Aeroporto Destino,Chegada Prevista,Chegada Real,Situação Voo,Justificativa,Referência,Situação Partida,Situação Chegada,arquivo_origem,linha_origem,arquivo_origem_mantido,linha_origem_mantida,motivo_descarte
0,AZU,AZUL LINHAS AÉREAS BRASILEIRAS S/A,5213,0,N,E195,118,SBSL,MARECHAL CUNHA MACHADO - SÃO LUÍS - MA - BRASIL,10/01/2020 22:00,10/01/2020 22:25,SBBE,INTERNACIONAL DE BELÉM/VAL DE CANS/JÚLIO CEZAR...,10/01/2020 23:05,10/01/2020 23:31,REALIZADO,AEROPORTO ORIGEM ABAIXO DOS LIMITES,2020-01-10,Pontual,Pontual,VRA_2020_01.csv,11522,VRA_2020_01.csv,11521,DUPLICATA_EXATA_20_COLUNAS
1,TCV,EMPRESA DE TRANSPORTES AÉREOS DE CABO VERDE - ...,0661,1,I,B752,183,GVAC,AEROPORTO INTERNACIONAL AMÍLCAR CABRAL - ILHA ...,22/01/2020 21:17,22/01/2020 21:17,SBSV,DEPUTADO LUÍS EDUARDO MAGALHÃES - SALVADOR - B...,23/01/2020 01:57,23/01/2020 01:57,REALIZADO,"ATRASOS NÃO ESPECÍFICOS, OUTROS",2020-01-22,Pontual,Pontual,VRA_2020_01.csv,90942,VRA_2020_01.csv,90941,DUPLICATA_EXATA_20_COLUNAS
2,TCV,EMPRESA DE TRANSPORTES AÉREOS DE CABO VERDE - ...,0662,0,I,B752,183,SBSV,DEPUTADO LUÍS EDUARDO MAGALHÃES - SALVADOR - B...,22/01/2020 23:30,,GVAC,AEROPORTO INTERNACIONAL AMÍLCAR CABRAL - ILHA ...,23/01/2020 04:45,,CANCELADO,CANCELAMENTO POR MOTIVOS TÉCNICOS - OPERACIONAIS,2020-01-22,,,VRA_2020_01.csv,90944,VRA_2020_01.csv,90943,DUPLICATA_EXATA_20_COLUNAS


### 4. Validar contagens, vínculos e preservação
As contagens esperadas são critérios desta entrega, não constantes usadas para filtrar os dados. Se a base mudar, as validações param a execução com uma mensagem de divergência.

Além de conferir `brutos = excedentes + deduplicados`, validamos que não restam duplicatas, que cada descarte tem um representante na saída e que seus 20 valores são idênticos aos da ocorrência mantida. Também conferimos que os descartes correspondem exatamente aos excedentes da base bruta e que a escolha respeitou a primeira ocorrência.

In [4]:
resumo_validacao = pd.DataFrame([
    {"indicador": "registros_brutos", "esperado": 483545, "observado": len(df_bruta)},
    {"indicador": "duplicatas_excedentes", "esperado": 124, "observado": len(df_duplicatas)},
    {"indicador": "apos_deduplicacao", "esperado": 483421, "observado": len(df_sem_duplicatas)},
])
resumo_validacao["confere"] = resumo_validacao["esperado"].eq(resumo_validacao["observado"])
display(resumo_validacao)
assert resumo_validacao["confere"].all(), "Contagens divergentes do diagnóstico de 2020."
assert len(df_bruta) == len(df_duplicatas) + len(df_sem_duplicatas)
assert not df_sem_duplicatas.duplicated(subset=colunas_originais).any()
assert df_sem_duplicatas.columns.tolist() == colunas_originais + colunas_origem

# A saída deve conter exatamente as primeiras ocorrências, sem modificar valores.
pd.testing.assert_frame_equal(df_sem_duplicatas, df_bruta.loc[~mascara_excedentes])
pd.testing.assert_frame_equal(
    df_duplicatas[df_bruta.columns].reset_index(drop=True),
    df_bruta.loc[mascara_excedentes].reset_index(drop=True),
)
assert df_duplicatas[["arquivo_origem_mantido", "linha_origem_mantida"]].notna().all().all()
assert not df_duplicatas.duplicated(subset=colunas_origem).any()

# Localiza cada representante na saída e compara todos os valores originais.
localizacoes_mantidas = pd.MultiIndex.from_frame(
    df_duplicatas[["arquivo_origem_mantido", "linha_origem_mantida"]]
    .set_axis(colunas_origem, axis="columns")
)
saida_por_origem = df_sem_duplicatas.set_index(colunas_origem)
assert localizacoes_mantidas.isin(saida_por_origem.index).all()
valores_mantidos = saida_por_origem.loc[localizacoes_mantidas, colunas_originais]
pd.testing.assert_frame_equal(
    df_duplicatas[colunas_originais].reset_index(drop=True),
    valores_mantidos.reset_index(drop=True),
)
assert assinatura_dataframe(df_bruta) == assinatura_bruta, "A base bruta em memória foi alterada."
assert {p.name: sha256_arquivo(p) for p in arquivos} == hashes_iniciais, "Um CSV original foi alterado."
print("Contagens, conteúdo, primeira ocorrência, auditoria e preservação validados.")

Contagens, conteúdo, primeira ocorrência, auditoria e preservação validados.


,indicador,esperado,observado,confere
0,registros_brutos,483545,483545,True
1,duplicatas_excedentes,124,124,True
2,apos_deduplicacao,483421,483421,True


### 5. Consolidar evidências mensais
A contagem de descartes é anual: se uma repetição ocorrer em arquivos diferentes, a ocorrência posterior também será auditada. O quadro mensal atribui cada descarte ao arquivo de sua ocorrência e deve reconciliar com o total anual.

In [5]:
resumo_mensal = resumo_arquivos[["arquivo", "registros_brutos"]].copy()
resumo_mensal["duplicatas_excedentes"] = (
    resumo_mensal["arquivo"].map(df_duplicatas["arquivo_origem"].value_counts())
    .fillna(0).astype(int)
)
resumo_mensal["apos_deduplicacao"] = (
    resumo_mensal["arquivo"].map(df_sem_duplicatas["arquivo_origem"].value_counts())
    .fillna(0).astype(int)
)
assert (resumo_mensal["registros_brutos"] ==
        resumo_mensal["duplicatas_excedentes"] + resumo_mensal["apos_deduplicacao"]).all()
assert resumo_mensal["duplicatas_excedentes"].sum() == len(df_duplicatas)
assert resumo_mensal["apos_deduplicacao"].sum() == len(df_sem_duplicatas)
display(resumo_mensal)
print("Descartes cujo representante está em outro arquivo:", int(
    df_duplicatas["arquivo_origem"].ne(df_duplicatas["arquivo_origem_mantido"]).sum()
))
print("Etapa 1 concluída: somente duplicatas exatas removidas da cópia; base bruta preservada.")

Descartes cujo representante está em outro arquivo: 0
Etapa 1 concluída: somente duplicatas exatas removidas da cópia; base bruta preservada.


,arquivo,registros_brutos,duplicatas_excedentes,apos_deduplicacao
0,VRA_2020_01.csv,92390,4,92386
1,VRA_2020_02.csv,80919,30,80889
2,VRA_2020_03.csv,64523,90,64433
3,VRA_2020_04.csv,7617,0,7617
4,VRA_2020_05.csv,9514,0,9514
5,VRA_2020_06.csv,12800,0,12800
6,VRA_2020_07.csv,19850,0,19850
7,VRA_2020_08.csv,24806,0,24806
8,VRA_2020_09.csv,30660,0,30660
9,VRA_2020_10.csv,40590,0,40590


### 6. Definir as regras da separação
A entrada é `df_sem_duplicatas`, para não contar excedentes também como rejeições. Aplicamos todas as regras de forma independente: um registro sem partida **e** chegada previstas recebe dois motivos; se também faltar companhia, recebe um terceiro.

| Campo obrigatório | Ausência | Validação de preenchidos |
|---|---|---|
| Sigla ICAO Empresa Aérea | nulo técnico, vazio ou somente espaços | três letras ASCII |
| Sigla ICAO Aeroporto Origem | nulo técnico, vazio ou somente espaços | quatro letras ASCII |
| Sigla ICAO Aeroporto Destino | nulo técnico, vazio ou somente espaços | quatro letras ASCII |
| Partida Prevista | nulo técnico, vazio ou somente espaços | `DD/MM/AAAA HH:MM` e calendário/horário válidos |
| Chegada Prevista | nulo técnico, vazio ou somente espaços | `DD/MM/AAAA HH:MM` e calendário/horário válidos |

O PDF determina rejeitar obrigatórios inválidos, mas não detalha um catálogo ICAO nem uma gramática de datas. Para operacionalizar essa regra, usamos a estrutura sondada na exploração: formato ICAO acima e datas operacionais observadas em 2019/2020. A validação de ICAO é **sintática**, sem consulta a cadastro oficial; elegibilidade aqui não garante a existência do código ou integridade referencial no futuro DW.

`strip()` e, nos códigos, `upper()` são usados somente nas cópias de validação, antecipando que espaços externos e caixa possam ser padronizados depois; não mudam os dados nem geram rejeição por si sós. Não usamos marcadores textuais como nulos automaticamente. Valores preenchidos incompatíveis com as regras recebem motivo de invalidade.

Datas reais, número do voo, justificativa, equipamento e demais campos não geram novas regras de rejeição nesta etapa. Voos cancelados com obrigatórios válidos continuam elegíveis mesmo sem horários reais. “Elegível” significa aprovado **nestas verificações**, sem transformação ou carga.

In [6]:
from datetime import datetime

assinatura_sem_duplicatas = assinatura_dataframe(df_sem_duplicatas)
assinatura_duplicatas = assinatura_dataframe(df_duplicatas)


def separar_elegiveis_rejeitados(base):
    """Classifica sem alterar a entrada e registra todas as violações de cada linha."""
    verificacoes = {}
    campos_por_motivo = {}

    def adicionar_regra(motivo, campo, mascara):
        verificacoes[motivo] = mascara.astype(bool)
        campos_por_motivo[motivo] = campo

    regras_icao = {
        "Sigla ICAO Empresa Aérea": ("COMPANHIA_ICAO", r"[A-Z]{3}"),
        "Sigla ICAO Aeroporto Origem": ("AEROPORTO_ORIGEM", r"[A-Z]{4}"),
        "Sigla ICAO Aeroporto Destino": ("AEROPORTO_DESTINO", r"[A-Z]{4}"),
    }
    regras_datas = {
        "Partida Prevista": "PARTIDA_PREVISTA",
        "Chegada Prevista": "CHEGADA_PREVISTA",
    }

    for campo, (prefixo, padrao) in regras_icao.items():
        texto = base[campo].astype("string").str.strip()
        ausente = texto.isna() | texto.eq("").fillna(False)
        formato_valido = texto.str.upper().str.fullmatch(padrao, na=False)
        adicionar_regra(f"{prefixo}_AUSENTE", campo, ausente)
        adicionar_regra(f"{prefixo}_FORMATO_INVALIDO", campo, ~ausente & ~formato_valido)

    def calendario_valido(valor):
        try:
            datetime.strptime(valor, "%d/%m/%Y %H:%M")
            return True
        except ValueError:
            return False

    for campo, prefixo in regras_datas.items():
        texto = base[campo].astype("string").str.strip()
        ausente = texto.isna() | texto.eq("").fillna(False)
        forma_valida = texto.str.fullmatch(
            r"[0-9]{2}/[0-9]{2}/[0-9]{4} [0-9]{2}:[0-9]{2}", na=False,
        )
        # Só valores distintos bem formados precisam da validação de calendário.
        distintos = texto.loc[~ausente & forma_valida].unique()
        datas_validas = {valor: calendario_valido(valor) for valor in distintos}
        data_valida = texto.map(datas_validas).eq(True)
        adicionar_regra(f"{prefixo}_AUSENTE", campo, ausente)
        adicionar_regra(f"{prefixo}_FORMATO_INVALIDO", campo, ~ausente & ~forma_valida)
        adicionar_regra(
            f"{prefixo}_CALENDARIO_HORARIO_INVALIDO", campo,
            ~ausente & forma_valida & ~data_valida,
        )

    falhas = pd.DataFrame(verificacoes, index=base.index)
    rejeitar = falhas.any(axis=1)
    elegiveis = base.loc[~rejeitar].copy()
    rejeitados = base.loc[rejeitar].copy()
    motivos = [tuple(linha.index[linha]) for _, linha in falhas.loc[rejeitar].iterrows()]
    rejeitados["motivos_rejeicao"] = pd.Series(motivos, index=rejeitados.index, dtype=object)
    rejeitados["quantidade_motivos"] = falhas.loc[rejeitar].sum(axis=1).astype(int)

    # Uma linha por motivo: permite filtrar, contar e auditar sem perder violações.
    detalhes = []
    for motivo, campo in campos_por_motivo.items():
        ocorrencias = base.loc[falhas[motivo], colunas_origem + [campo]].copy()
        ocorrencias = ocorrencias.rename(columns={campo: "valor_original"})
        ocorrencias["campo"] = campo
        ocorrencias["motivo_rejeicao"] = motivo
        detalhes.append(ocorrencias)
    motivos_rejeicao = pd.concat(detalhes, ignore_index=True)[
        colunas_origem + ["campo", "valor_original", "motivo_rejeicao"]
    ]
    return elegiveis, rejeitados, motivos_rejeicao, falhas


df_elegiveis, df_rejeitados, df_motivos_rejeicao, df_falhas_obrigatorios = (
    separar_elegiveis_rejeitados(df_sem_duplicatas)
)
print("Elegíveis:", len(df_elegiveis))
print("Rejeitados:", len(df_rejeitados))
print("Ocorrências de motivos de rejeição:", len(df_motivos_rejeicao))
display(df_rejeitados[
    colunas_origem + ["Partida Prevista", "Chegada Prevista", "motivos_rejeicao", "quantidade_motivos"]
].head(10))
display(df_motivos_rejeicao.head(10))

Elegíveis: 481909
Rejeitados: 1512
Ocorrências de motivos de rejeição: 3024


,arquivo_origem,linha_origem,Partida Prevista,Chegada Prevista,motivos_rejeicao,quantidade_motivos
212909,VRA_2020_03.csv,39602,,,"(PARTIDA_PREVISTA_AUSENTE, CHEGADA_PREVISTA_AU...",2
212910,VRA_2020_03.csv,39603,,,"(PARTIDA_PREVISTA_AUSENTE, CHEGADA_PREVISTA_AU...",2
212911,VRA_2020_03.csv,39604,,,"(PARTIDA_PREVISTA_AUSENTE, CHEGADA_PREVISTA_AU...",2
212912,VRA_2020_03.csv,39605,,,"(PARTIDA_PREVISTA_AUSENTE, CHEGADA_PREVISTA_AU...",2
212913,VRA_2020_03.csv,39606,,,"(PARTIDA_PREVISTA_AUSENTE, CHEGADA_PREVISTA_AU...",2
242407,VRA_2020_04.csv,4577,,,"(PARTIDA_PREVISTA_AUSENTE, CHEGADA_PREVISTA_AU...",2
242414,VRA_2020_04.csv,4584,,,"(PARTIDA_PREVISTA_AUSENTE, CHEGADA_PREVISTA_AU...",2
242431,VRA_2020_04.csv,4601,,,"(PARTIDA_PREVISTA_AUSENTE, CHEGADA_PREVISTA_AU...",2
242454,VRA_2020_04.csv,4624,,,"(PARTIDA_PREVISTA_AUSENTE, CHEGADA_PREVISTA_AU...",2
242485,VRA_2020_04.csv,4655,,,"(PARTIDA_PREVISTA_AUSENTE, CHEGADA_PREVISTA_AU...",2


,arquivo_origem,linha_origem,campo,valor_original,motivo_rejeicao
0,VRA_2020_03.csv,39602,Partida Prevista,,PARTIDA_PREVISTA_AUSENTE
1,VRA_2020_03.csv,39603,Partida Prevista,,PARTIDA_PREVISTA_AUSENTE
2,VRA_2020_03.csv,39604,Partida Prevista,,PARTIDA_PREVISTA_AUSENTE
3,VRA_2020_03.csv,39605,Partida Prevista,,PARTIDA_PREVISTA_AUSENTE
4,VRA_2020_03.csv,39606,Partida Prevista,,PARTIDA_PREVISTA_AUSENTE
5,VRA_2020_04.csv,4577,Partida Prevista,,PARTIDA_PREVISTA_AUSENTE
6,VRA_2020_04.csv,4584,Partida Prevista,,PARTIDA_PREVISTA_AUSENTE
7,VRA_2020_04.csv,4601,Partida Prevista,,PARTIDA_PREVISTA_AUSENTE
8,VRA_2020_04.csv,4624,Partida Prevista,,PARTIDA_PREVISTA_AUSENTE
9,VRA_2020_04.csv,4655,Partida Prevista,,PARTIDA_PREVISTA_AUSENTE


### 7. Validar a partição e a auditoria de todos os motivos
Cada linha da base deduplicada deve estar em exatamente uma saída. As 20 colunas originais, os dois metadados e os índices são preservados; isso permite reconstruir integralmente `df_sem_duplicatas` juntando as saídas. Rejeitados recebem apenas as duas colunas técnicas `motivos_rejeicao` e `quantidade_motivos`.

A auditoria detalhada registra arquivo, posição lógica, campo, valor original e um código por motivo. A soma das contagens por motivo pode ser maior que o número de rejeitados. Validamos também que todos os motivos marcados aparecem nessa auditoria e que nenhuma duplicata excedente foi contada como rejeição.

In [7]:
mascara_rejeicao = df_falhas_obrigatorios.any(axis=1)
assert df_elegiveis.index.intersection(df_rejeitados.index).empty
assert len(df_sem_duplicatas) == len(df_elegiveis) + len(df_rejeitados)
assert len(df_bruta) == len(df_duplicatas) + len(df_rejeitados) + len(df_elegiveis)
assert not df_falhas_obrigatorios.loc[df_elegiveis.index].any().any()
assert df_falhas_obrigatorios.loc[df_rejeitados.index].any(axis=1).all()
pd.testing.assert_frame_equal(df_elegiveis, df_sem_duplicatas.loc[~mascara_rejeicao])
pd.testing.assert_frame_equal(
    df_rejeitados[df_sem_duplicatas.columns], df_sem_duplicatas.loc[mascara_rejeicao],
)
reconstruida = pd.concat([
    df_elegiveis, df_rejeitados[df_sem_duplicatas.columns],
]).sort_index()
pd.testing.assert_frame_equal(reconstruida, df_sem_duplicatas)
del reconstruida

assert df_rejeitados["quantidade_motivos"].eq(df_rejeitados["motivos_rejeicao"].map(len)).all()
assert df_rejeitados["quantidade_motivos"].ge(1).all()
assert len(df_motivos_rejeicao) == int(df_falhas_obrigatorios.sum().sum())
assert len(df_motivos_rejeicao) == int(df_rejeitados["quantidade_motivos"].sum())
assert not df_motivos_rejeicao.duplicated(colunas_origem + ["motivo_rejeicao"]).any()

rejeitados_por_origem = df_rejeitados.set_index(colunas_origem)
assert len(df_motivos_rejeicao[colunas_origem].drop_duplicates()) == len(df_rejeitados)
for motivo in df_falhas_obrigatorios.columns:
    detalhes = df_motivos_rejeicao.loc[df_motivos_rejeicao["motivo_rejeicao"].eq(motivo)]
    campo = detalhes["campo"].iloc[0] if not detalhes.empty else None
    if campo is not None:
        esperados = df_sem_duplicatas.loc[df_falhas_obrigatorios[motivo], colunas_origem + [campo]]
        pd.testing.assert_frame_equal(
            detalhes[colunas_origem + ["valor_original"]].reset_index(drop=True),
            esperados.rename(columns={campo: "valor_original"}).reset_index(drop=True),
        )
    assert len(detalhes) == int(df_falhas_obrigatorios[motivo].sum())

motivos_agrupados = df_motivos_rejeicao.groupby(colunas_origem)["motivo_rejeicao"].agg(tuple)
pd.testing.assert_series_equal(
    motivos_agrupados.sort_index(), rejeitados_por_origem["motivos_rejeicao"].sort_index(),
    check_names=False,
)
chaves_descartadas = pd.MultiIndex.from_frame(df_duplicatas[colunas_origem])
assert not chaves_descartadas.isin(rejeitados_por_origem.index).any()
assert assinatura_dataframe(df_bruta) == assinatura_bruta
assert assinatura_dataframe(df_sem_duplicatas) == assinatura_sem_duplicatas
assert assinatura_dataframe(df_duplicatas) == assinatura_duplicatas
assert {p.name: sha256_arquivo(p) for p in arquivos} == hashes_iniciais

# A exploração previa essas contagens; os motivos continuam sendo medidos pelas regras.
assert len(df_rejeitados) == 1512, "Rejeições divergentes da exploração: revise os motivos."
assert len(df_elegiveis) == 481909, "Elegíveis divergentes da exploração: revise os motivos."
resumo_separacao = pd.DataFrame([
    ("brutos", len(df_bruta)), ("duplicatas_excedentes", len(df_duplicatas)),
    ("apos_deduplicacao", len(df_sem_duplicatas)), ("rejeitados", len(df_rejeitados)),
    ("elegiveis", len(df_elegiveis)), ("ocorrencias_de_motivos", len(df_motivos_rejeicao)),
], columns=["indicador", "registros"])
display(resumo_separacao)
print("Partição, reconciliação, todos os motivos e preservação validados.")

Partição, reconciliação, todos os motivos e preservação validados.


,indicador,registros
0,brutos,483545
1,duplicatas_excedentes,124
2,apos_deduplicacao,483421
3,rejeitados,1512
4,elegiveis,481909
5,ocorrencias_de_motivos,3024


### 8. Evidências por motivo e por mês
As contagens incluem todas as regras, inclusive as que não encontraram violações. A distribuição de combinações mostra quantos motivos cada registro acumula. As saídas detalhadas ficam nos DataFrames reproduzíveis; nesta entrega não exportamos arquivos nem carregamos banco.

In [8]:
resumo_motivos = df_falhas_obrigatorios.sum().rename_axis("motivo_rejeicao").reset_index(name="ocorrencias")
display(resumo_motivos)
display(df_rejeitados["motivos_rejeicao"].value_counts().rename_axis("combinacao_de_motivos")
        .reset_index(name="registros"))
resumo_mensal_separacao = resumo_mensal.copy()
for rotulo, quadro in [("rejeitados", df_rejeitados), ("elegiveis", df_elegiveis)]:
    resumo_mensal_separacao[rotulo] = (
        resumo_mensal_separacao["arquivo"].map(quadro["arquivo_origem"].value_counts())
        .fillna(0).astype(int)
    )
assert resumo_mensal_separacao["apos_deduplicacao"].eq(
    resumo_mensal_separacao["rejeitados"] + resumo_mensal_separacao["elegiveis"]
).all()
display(resumo_mensal_separacao)
print("Separação concluída. Valores originais preservados; nenhum cálculo de métricas ou carga executado.")

Separação concluída. Valores originais preservados; nenhum cálculo de métricas ou carga executado.


,motivo_rejeicao,ocorrencias
0,COMPANHIA_ICAO_AUSENTE,0
1,COMPANHIA_ICAO_FORMATO_INVALIDO,0
2,AEROPORTO_ORIGEM_AUSENTE,0
3,AEROPORTO_ORIGEM_FORMATO_INVALIDO,0
4,AEROPORTO_DESTINO_AUSENTE,0
5,AEROPORTO_DESTINO_FORMATO_INVALIDO,0
6,PARTIDA_PREVISTA_AUSENTE,1512
7,PARTIDA_PREVISTA_FORMATO_INVALIDO,0
8,PARTIDA_PREVISTA_CALENDARIO_HORARIO_INVALIDO,0
9,CHEGADA_PREVISTA_AUSENTE,1512


,combinacao_de_motivos,registros
0,"(PARTIDA_PREVISTA_AUSENTE, CHEGADA_PREVISTA_AU...",1512


,arquivo,registros_brutos,duplicatas_excedentes,apos_deduplicacao,rejeitados,elegiveis
0,VRA_2020_01.csv,92390,4,92386,0,92386
1,VRA_2020_02.csv,80919,30,80889,0,80889
2,VRA_2020_03.csv,64523,90,64433,5,64428
3,VRA_2020_04.csv,7617,0,7617,66,7551
4,VRA_2020_05.csv,9514,0,9514,70,9444
5,VRA_2020_06.csv,12800,0,12800,164,12636
6,VRA_2020_07.csv,19850,0,19850,253,19597
7,VRA_2020_08.csv,24806,0,24806,259,24547
8,VRA_2020_09.csv,30660,0,30660,242,30418
9,VRA_2020_10.csv,40590,0,40590,135,40455


### Resultados e limites

- `df_bruta`: todas as ocorrências recebidas, com arquivo e posição lógica de origem.
- `df_sem_duplicatas`: primeiras ocorrências exatas, sem outras remoções ou conversões.
- `df_duplicatas`: descartes por repetição e a localização da ocorrência mantida.
- `df_elegiveis`: linhas aprovadas nas cinco verificações obrigatórias, com os 20 valores originais e a rastreabilidade.
- `df_rejeitados`: linhas reprovadas, preservando os valores e acrescentando todos os motivos e sua quantidade.
- `df_motivos_rejeicao`: auditoria com uma linha por motivo, incluindo campo e valor original.

As saídas acima são reproduzíveis pela execução do notebook. A conferência por SHA-256 garante que os CSVs não foram alterados durante a execução; a reconstrução das partições garante preservação dos dados em memória. A elegibilidade não verifica cadastro ICAO, resolução de dimensões, convenção de fuso ou outras regras que dependem de confirmação do grupo.

A modelagem em PDF foi consultada para esta separação. Contrato Markdown e modelo Draw.io não estão disponíveis. As etapas futuras de transformação, métricas, dimensões e carga PostgreSQL permanecem fora desta entrega. Nenhum outro notebook deve ser alterado.